# Reproducing the MuVIcell tutorial with MINA

[MuVIcell](https://github.com/HartmannLab/MuVIcell) demonstrated *multicellular
factor analysis*: starting from pseudo-bulked, cell-type–stratified features, it
used **MuVI** to learn latent factors that capture coordinated programs across
cell types, and then related those factors to sample-level covariates.

**MINA** generalises this workflow. The factor-decomposition step is more
flexible — MINA builds on [MOFA-FLEX](https://github.com/bioFAM/mofaflex), which
supports several priors and likelihoods — while the downstream statistics and
visualisations mirror (and extend) MuVIcell. This notebook reproduces the
MuVIcell tutorial end to end with MINA.

!!! note "About the data"
    MuVIcell's *synthetic data generator* is **not** part of MINA. We ran
    MuVIcell's own `synthetic.generate_synthetic_data` + `add_latent_structure`
    **once** and exported the resulting object to
    `data/muvicell_synthetic.h5mu` (200 samples × 3 cell-type views, a known
    3-factor latent structure, and an ordinal `exposure` covariate correlated
    with factor 1). The example loads that object directly.

!!! tip "Requirements"
    `pip install "mina[spatial]"` (for the plotting helpers) and a working
    `mofaflex` install (Python ≥ 3.11).

In [ ]:
import numpy as np
import pandas as pd
import mudata as md
import matplotlib.pyplot as plt

import mofaflex as mf
import mina

## 1. Load the pseudo-bulked multi-view object

MINA operates on a *dictionary of per-view `AnnData` objects* together with a
sample-level metadata table (one row per sample).

In [ ]:
mdata = md.read("data/muvicell_synthetic.h5mu")

anndata_dict = {view: mdata[view].copy() for view in mdata.mod}

metadata = mdata.obs.copy()
metadata["sample_id"] = metadata.index.astype(str)
mdata

## 2. Preprocessing

MuVIcell centred and scaled each view independently. MINA reproduces this exact
behaviour with `norm_log(method="zscore")`. We then prefix each feature with its
view name so the model and the downstream helpers can reconstruct view-specific
loadings.

In [ ]:
mina.up.norm_log(anndata_dict, method="zscore")
mina.up.utils.append_view_to_var(anndata_dict)

## 3. Factor decomposition (MOFA-FLEX)

Where MuVIcell called MuVI, MINA uses the flexible MOFA-FLEX engine. We request
3 factors to match the injected latent structure and use a spike-and-slab weight
prior for sparse, interpretable loadings.

In [ ]:
mdata_model = md.MuData(anndata_dict)

model = mf.terms.MofaFlex(
    n_factors=3,
    weight_prior="SpikeSlab",
    init_factors="pca",
)
model.fit(
    mdata_model,
    seed=42,
    lr=0.01,
    early_stopper_patience=1000,
    subset_var=None,
    save_path=False,
    likelihoods="Normal",
)

## 4. Assemble a model `AnnData`

`model_to_anndata` collects factor scores (`.X`), explained variance per factor
and view (`.var`), gene loadings (`.varm`) and the aligned pseudo-bulk matrices
(`.obsm`) into a single annotated object that all downstream functions consume.

In [ ]:
amodel = mina.down.model_to_anndata(
    anndata_dict=anndata_dict,
    metadata=metadata,
    model=model,
)
amodel

## 5. Variance explained and reconstruction quality

In [ ]:
variance_df = mina.down.variance_by_view_info(amodel)
mina.pl.plot_variance_by_view(variance_df)

In [ ]:
reconstruction = mina.down.reconstruction_info(amodel)
print("Macro R2:", round(reconstruction["macro"]["R2"], 3))
mina.pl.plot_reconstruction(reconstruction["by_view"])

## 6. Associating factors with covariates

The tutorial injected an ordinal `exposure` variable correlated with factor 1.
MINA offers a Kruskal–Wallis test (categorical groups) and a Kendall tau test
(ordinal variables), both with optional Bonferroni correction.

In [ ]:
scores = mina.down.factor_scores_info(amodel, obs_keys=["exposure", "batch"])

kruskal_df = mina.down.kruskal_info(scores, group_col="exposure")
kruskal_df

In [ ]:
scores["exposure_ordinal"] = scores["exposure"].cat.codes
mina.down.kendall_info(scores, ordinal_col="exposure_ordinal")

In [ ]:
top_factor = kruskal_df.iloc[0]["factor"]
mina.pl.plot_factor_violin(scores, factor=top_factor, group_col="exposure")

A compact way to screen many covariates at once is the adjusted p-value
matrix (factors × covariates).

In [ ]:
pval_matrix = mina.down.get_pval_matrix(amodel, ["exposure", "batch"])
mina.pl.plot_pval_tiles(pval_matrix, title="Factor-covariate associations", star_threshold=0.05)
plt.show()

## 7. Loadings and top features

In [ ]:
loadings = mina.down.variable_loadings_info(amodel)
mina.pl.plot_top_loadings_heatmap(loadings, factor=top_factor, top_n=20)

In [ ]:
top_features = mina.down.top_features_by_view_info(
    loadings, factors=[top_factor], top_per_view=3
)
top_features

## 8. Sample structure in factor space

Confidence ellipses summarise how groups separate along a pair of factors.

In [ ]:
ellipse_df = mina.down.confidence_ellipses_info(
    scores, x_factor="Factor1", y_factor="Factor2", group_col="exposure"
)
mina.pl.plot_confidence_ellipses(
    scores, ellipse_df, x_factor="Factor1", y_factor="Factor2", group_col="exposure"
)

## 9. Exporting selected features

Mirroring MuVIcell's final step, `build_selected_anndata` materialises a small
`AnnData` containing only the selected (feature, view) pairs — handy for
downstream visualisation or sharing.

In [ ]:
selected = mina.down.build_selected_anndata(amodel, top_features)
selected

## Summary

Using MINA we reproduced the MuVIcell tutorial: per-view z-scoring, a 3-factor
decomposition (via MOFA-FLEX), variance-explained and reconstruction summaries,
factor–covariate association tests, loading inspection and confidence ellipses.
The only conceptual change is the decomposition engine — everything downstream
follows the same logic, now available through a single `mina.down` / `mina.pl`
API.

See *Best practices for spatial proteomics* for guidance on applying this
workflow to real imaging data, and the *MIBI colorectal cancer* case study for a
full real-data example.